# Phase 1 - Raw Data Ingestion

### Set up the medallion layers
- Three schemas — `bronze`, `silver`, `gold` — one per layer, so trust level is visible right in the catalog.
- **Schema** = a folder for tables. **Volume** = a folder for files (where the raw CSVs land).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;
CREATE VOLUME  IF NOT EXISTS workspace.bronze.raw;

### Ingest raw CSVs into Bronze — copied, never changed
- Bronze is a faithful copy of the source; we alter no values, only add two audit columns.
- **`ingestion_timestamp`** = when we loaded it; **`source_file`** = which file it came from → every row is traceable.
- **Delta table** = Parquet files + a transaction log → reliable writes, schema tracking, time-travel.

In [0]:
from pyspark.sql.functions import current_timestamp, lit
RAW = "/Volumes/workspace/bronze/raw"

In [0]:
def ingest_to_bronze(file_name, table_name):
    df = (spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(f"{RAW}/{file_name}"))
    df = (df
            .withColumn("ingestion_timestamp", current_timestamp())  # audit col 1
            .withColumn("source_file", lit(file_name)))              # audit col 2
    (df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.bronze.{table_name}"))
    print(f"{table_name}: {df.count()} rows -> workspace.bronze.{table_name}")
    return df

In [0]:
ingest_to_bronze("olist_orders_dataset.csv",     "orders")
ingest_to_bronze("olist_customers_dataset.csv",  "customers")
ingest_to_bronze("olist_products_dataset.csv",   "products")
ingest_to_bronze("olist_order_items_dataset.csv","order_items")   # needed for Phase 2 revenue

orders: 99441 rows -> workspace.bronze.orders
customers: 99441 rows -> workspace.bronze.customers
products: 32951 rows -> workspace.bronze.products
order_items: 112650 rows -> workspace.bronze.order_items


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double, ingestion_timestamp: timestamp, source_file: string]

In [0]:
spark.table("workspace.bronze.orders").printSchema()
spark.sql("DESCRIBE TABLE workspace.bronze.orders").display()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_status,string,null
order_purchase_timestamp,timestamp,null
order_approved_at,timestamp,null
order_delivered_carrier_date,timestamp,null
order_delivered_customer_date,timestamp,null
order_estimated_delivery_date,timestamp,null
ingestion_timestamp,timestamp,null
source_file,string,null
